In [0]:
#dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

In [0]:
import json
from datetime import date

from common_utils.bronze import read_raw, bronze_ingestor
from common_utils.logging import get_logger


# ============================================================
# 0. LOGGER
# ============================================================

logger = get_logger("ds2b")


# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

logger.info("Reading Bronze configuration from %s", config_path)


# ============================================================
# 2. READ CONFIG
# ============================================================

with open(config_path, "r") as f:
    config = json.load(f)


# ============================================================
# 3. RUN DATE
# ============================================================

run_date = date.today().isoformat()

logger.info("Bronze ingestion run date: %s", run_date)


# ============================================================
# 4. BASE RAW PATH
# ============================================================

base_raw_path = "/Volumes/retaildataplatform/bronze/raw_data"

logger.info("Base raw path: %s", base_raw_path)


# ============================================================
# 5. PROCESS EACH SOURCE
# ============================================================

for source_name, source_config in config.items():

    raw_folder = source_config["raw_folder"]
    file_format = source_config["file_format"]
    read_options = source_config.get("read_options", {})
    target_table = source_config["target_table"]
    mode = source_config["mode"]

    # --------------------------------------------------------
    # Build raw path
    # --------------------------------------------------------

    raw_path = (
        f"{base_raw_path}/"
        f"{raw_folder}/"
        f"load_date={run_date}"
    )

    logger.info(
        "[%s] Starting Bronze ingestion",
        source_name
    )

    logger.info(
        "[%s] Reading raw data from %s",
        source_name,
        raw_path
    )

    # --------------------------------------------------------
    # Read Raw
    # --------------------------------------------------------

    df = read_raw(
        spark,
        raw_path,
        file_format=file_format,
        options=read_options
    )

    logger.info(
        "[%s] Raw data successfully read",
        source_name
    )

    # --------------------------------------------------------
    # Bronze ingestion
    # --------------------------------------------------------

    rows = bronze_ingestor(
        df,
        mode=mode,
        target_table=target_table
    )

    logger.info(
        "[%s] Successfully ingested %s rows into %s",
        source_name,
        rows,
        target_table
    )


# ============================================================
# 6. COMPLETION
# ============================================================

logger.info("Bronze ingestion finished successfully for all sources")